# GPT-2 Softmax vs Linear Attention 실험 (Colab)

`Gpt2_exp_replacement_guide.md` / `Colab_Drive_Integration.md` 참고.

**런타임 > 런타임 유형 변경 > GPU(T4)**로 설정한 뒤, 위에서부터 순서대로 실행하세요.
세션이 끊기면 "세션이 끊겼다면" 섹션만 다시 실행하면 이어서 진행됩니다.

## 0. GPU 확인

In [ ]:
import torch

print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

## 1. Google Drive 마운트

체크포인트·데이터 캐시·로그·결과를 세션이 끊겨도 남도록 Drive에 저장한다.

In [ ]:
from google.colab import drive

drive.mount("/content/drive")

## 2. 코드 준비

`REPO_URL`을 본인 레포 주소로 바꾼다.

In [ ]:
REPO_URL = "<YOUR_REPO_URL>"  # 예: https://github.com/<user>/<repo>.git

!git clone {REPO_URL} repo
%cd repo/llm/sqrt_remove
!pip install -q -r requirements.txt

## 3. 경로 상수 (전부 Drive 하위)

In [ ]:
import os

DRIVE_ROOT = "/content/drive/MyDrive/gpt2_exp"
CACHE_DIR = f"{DRIVE_ROOT}/data_cache"
LOG_DIR = f"{DRIVE_ROOT}/logs"
RESULTS_DIR = f"{DRIVE_ROOT}/results"

# 실험 단계별로 체크포인트 디렉터리를 분리한다. 같은 디렉터리를 공유하면
# --resume이 다른 규모의 모델 체크포인트를 불러오려다 shape mismatch로 실패한다
# (1단계는 n_layer=4/n_embd=128 소형 모델, 2단계는 n_layer=12/n_embd=768 GPT-2 small).
PIPELINE_CKPT_DIR = f"{DRIVE_ROOT}/checkpoints/pipeline_check"
MAIN_CKPT_DIR = f"{DRIVE_ROOT}/checkpoints/main"

for d in (CACHE_DIR, PIPELINE_CKPT_DIR, MAIN_CKPT_DIR, LOG_DIR, RESULTS_DIR):
    os.makedirs(d, exist_ok=True)

print("DRIVE_ROOT:", DRIVE_ROOT)


## 4. 1단계: 파이프라인 검증 (TinyShakespeare, 소규모)

본 실험 전에 attention 스위치·loss 감소 여부를 빠르게 확인한다 (가이드 1단계).

In [ ]:
!python -m gpt2.train \
  --attention-type softmax \
  --dataset tiny_shakespeare \
  --n-layer 4 --n-head 4 --n-embd 128 --block-size 128 \
  --batch-size 8 --max-steps 200 \
  --eval-interval 50 --ckpt-interval 50 \
  --cache-dir "{CACHE_DIR}" \
  --ckpt-dir "{PIPELINE_CKPT_DIR}" \
  --log-path "{LOG_DIR}/pipeline_check_softmax.csv" \
  --resume


In [ ]:
!python -m gpt2.train \
  --attention-type linear \
  --dataset tiny_shakespeare \
  --n-layer 4 --n-head 4 --n-embd 128 --block-size 128 \
  --batch-size 8 --max-steps 200 \
  --eval-interval 50 --ckpt-interval 50 \
  --cache-dir "{CACHE_DIR}" \
  --ckpt-dir "{PIPELINE_CKPT_DIR}" \
  --log-path "{LOG_DIR}/pipeline_check_linear.csv" \
  --resume


## 5. 2단계: 본 실험 학습 (GPT-2 small, WikiText-103)

baseline(softmax) -> variant(linear) 순서로 **같은 세션에서 연달아** 실행한다
(GPU 배정 편차 최소화, 가이드 4장). `--save-every-epoch`로 epoch마다 별도 체크포인트도 남긴다.

**메모리**: mixed precision(bf16/fp16)은 CUDA에서 기본으로 켜진다. 그래도 T4(16GB)에서
`CUDA out of memory`가 나면:
1. `--batch-size`를 낮추고 `--grad-accum-steps`를 그만큼 올려 유효 batch size를 유지한다
   (아래 기본값은 batch-size 8 x grad-accum-steps 2 = 유효 batch 16).
2. 그래도 부족하면 `--grad-checkpointing`을 추가한다 (속도는 느려지지만 메모리를 더 아낀다).

세션이 12시간 제한으로 끊기면, 이 두 셀을 그대로 다시 실행하면 `--resume` 덕분에
중단된 지점부터 이어진다.


In [ ]:
!python -m gpt2.train \
  --attention-type softmax \
  --dataset wikitext-103 \
  --n-layer 12 --n-head 12 --n-embd 768 --block-size 512 \
  --batch-size 8 --grad-accum-steps 2 --max-steps 5000 \
  --eval-interval 200 --ckpt-interval 200 \
  --save-every-epoch \
  --cache-dir "{CACHE_DIR}" \
  --ckpt-dir "{MAIN_CKPT_DIR}" \
  --log-path "{LOG_DIR}/softmax.csv" \
  --resume
# OOM이 계속되면 위 명령에 --grad-checkpointing 을 추가하거나 --batch-size 4 --grad-accum-steps 4 로 낮추세요.


In [ ]:
!python -m gpt2.train \
  --attention-type linear \
  --dataset wikitext-103 \
  --n-layer 12 --n-head 12 --n-embd 768 --block-size 512 \
  --batch-size 8 --grad-accum-steps 2 --max-steps 5000 \
  --eval-interval 200 --ckpt-interval 200 \
  --save-every-epoch \
  --cache-dir "{CACHE_DIR}" \
  --ckpt-dir "{MAIN_CKPT_DIR}" \
  --log-path "{LOG_DIR}/linear.csv" \
  --resume
# OOM이 계속되면 위 명령에 --grad-checkpointing 을 추가하거나 --batch-size 4 --grad-accum-steps 4 로 낮추세요.


## 6. 3단계: 벤치마크

block_size(시퀀스 길이)를 단계적으로 바꿔가며 softmax/linear를 연달아 측정한다
(가설 H1·H3: 시퀀스가 길어질수록 linear의 속도 이점이 커지는지 확인).

mixed precision은 기본으로 켜진다. block_size=1024처럼 큰 값에서 `CUDA out of memory`가
나면 `--batch-size`를 낮추거나 `--grad-checkpointing`을 추가한다 (아래 셀 마지막 줄 참고).


In [ ]:
!python -m gpt2.benchmark \
  --dataset wikitext-103 \
  --n-layer 12 --n-head 12 --n-embd 768 \
  --block-sizes 256 512 1024 \
  --seeds 1337 42 7 \
  --batch-size 8 \
  --cache-dir "{CACHE_DIR}" \
  --output "{RESULTS_DIR}/benchmark_results.json"
# block_size=1024에서 OOM이면: --batch-size 4 를 쓰거나 --grad-checkpointing 을 추가하세요.


## 7. 4단계: 결과 집계

`gpt2/aggregate_results.py`가 seed들을 평균 내서 가이드 7장 템플릿 표와
softmax/linear 속도·정확도 비교 표를 markdown으로 생성한다.

In [ ]:
!python -m gpt2.aggregate_results \
  --input "{RESULTS_DIR}/benchmark_results.json" \
  --output "{RESULTS_DIR}/results_table.md"

In [ ]:
from IPython.display import Markdown, display

with open(f"{RESULTS_DIR}/results_table.md") as f:
    display(Markdown(f.read()))

## 세션이 끊겼다면

1. 런타임을 다시 연결한다.
2. **0(GPU 확인) → 1(Drive 마운트) → 2(코드 준비) → 3(경로 상수)** 셀을 다시 실행한다
   (`/content/`는 초기화되지만 Drive 내용은 그대로 남아있다).
3. 중단됐던 학습/벤치마크 셀을 **그대로 다시 실행**한다. `--resume` 플래그 덕분에
   `{PIPELINE_CKPT_DIR 또는 MAIN_CKPT_DIR}/{attention_type}/latest.pt`에 저장된 step부터
   자동으로 이어진다.
4. 체크포인트가 쌓였는지 확인하려면:
   ```python
   !ls -la "{MAIN_CKPT_DIR}/softmax"
   ```

> 주의: 1단계(파이프라인 검증)와 2단계(본 실험)는 모델 규모가 다르므로
> **절대 같은 --ckpt-dir을 공유하면 안 된다** (`PIPELINE_CKPT_DIR` vs `MAIN_CKPT_DIR`로
> 이미 분리되어 있음). 이 둘을 섞으면 `--resume` 시 `size mismatch` 에러가 난다.
